# ECS 171 Group Project Team 9

## Introduction

Heart disease is one of the leading causes of death worldwide. Early prediction of heart disease risk can support timely medical intervention and lifestyle changes. The goal of this project is to develop and evaluate machine learning models that predict whether a patient is at risk of heart disease based on clinical and demographic attributes.

This is formulated as a binary classification problem, where the output indicates the presence or absence of heart disease.

## Dataset Description

Dataset: UCI Heart Disease Dataset

Size: ~300 instances
Attributes: ~13 features
Target variable: Presence of heart disease (0 = No, 1 = Yes)

## Exploratory Data Analysis (EDA)

### Data Input

In [50]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [51]:
from ucimlrepo import fetch_ucirepo

heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

print(heart_disease.metadata)
print(heart_disease.variables)

{'uci_id': 45, 'name': 'Heart Disease', 'repository_url': 'https://archive.ics.uci.edu/dataset/45/heart+disease', 'data_url': 'https://archive.ics.uci.edu/static/public/45/data.csv', 'abstract': '4 databases: Cleveland, Hungary, Switzerland, and the VA Long Beach', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 303, 'num_features': 13, 'feature_types': ['Categorical', 'Integer', 'Real'], 'demographics': ['Age', 'Sex'], 'target_col': ['num'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1989, 'last_updated': 'Fri Nov 03 2023', 'dataset_doi': '10.24432/C52P4X', 'creators': ['Andras Janosi', 'William Steinbrunn', 'Matthias Pfisterer', 'Robert Detrano'], 'intro_paper': {'ID': 231, 'type': 'NATIVE', 'title': 'International application of a new probability algorithm for the diagnosis of coronary artery disease.', 'authors': 'R. Detrano, A. Jánosi, W. Steinbrunn, M

In [52]:
X = heart_disease.data.features.copy()
y = heart_disease.data.targets.copy()
X.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0


In [53]:
y.head()

,num
0,0
1,2
2,1
3,0
4,0


### Data Cleaning

#### Finding Missing Values and Replace it with NaN

In [54]:
import numpy as np
import pandas as pd

X = X.replace('?', np.nan)
X.isna().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
dtype: int64

#### Turning Data Types to Numeric

In [55]:
num_cols = ['age','trestbps','chol','thalach','oldpeak','ca']
X[num_cols] = X[num_cols].apply(pd.to_numeric)

cat_cols = ['sex','cp','fbs','restecg','exang','slope','thal']
X[cat_cols] = X[cat_cols].apply(pd.to_numeric)

X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
dtypes: float64(3), int64(10)
memory usage: 30.9 KB


#### Turn NaN into numbers

In [56]:
X['ca'] = X['ca'].fillna(X['ca'].median())
X['thal'] = X['thal'].fillna(X['thal'].mode()[0])

X.isna().sum()

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
dtype: int64

In [57]:
import os
df = pd.concat([X, y], axis=1)
df.rename(columns={'num':'target'}, inplace=True)
os.makedirs("data", exist_ok=True)
df.to_csv("heart_disease_raw.csv", index=False)

#### Outlier Detection

In [58]:
def detect_outliers(df, columns):
    outlier_indices = []
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower) | (df[col] > upper)].index
        outlier_indices.extend(outliers)

    return list(set(outlier_indices))


In [59]:
outlier_cols = ['age','trestbps','chol','thalach','oldpeak']
outliers = detect_outliers(X, outlier_cols)

len(outliers)

19

#### Cap Outlier

In [60]:
def cap_outliers(df, columns):
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        df[col] = np.where(df[col] < lower, lower, df[col])
        df[col] = np.where(df[col] > upper, upper, df[col])
        
    return df

X = cap_outliers(X, outlier_cols)

#### Change Target Value from severity of hear disease to if the patient has heart disease or not

In [61]:
y = (y['num'] > 0).astype(int)

In [62]:
y.head()

0    0
1    1
2    1
3    0
4    0
Name: num, dtype: int32

#### Encode Categorical Variables

In [63]:
X = pd.get_dummies(X, columns=['cp','restecg','slope','thal'], drop_first=True)
X.head()

,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,ca,cp_2,cp_3,cp_4,restecg_1,restecg_2,slope_2,slope_3,thal_6.0,thal_7.0
0,63.0,1,145.0,233.0,1,150.0,0,2.3,0.0,False,False,False,False,True,False,True,True,False
1,67.0,1,160.0,286.0,0,108.0,1,1.5,3.0,False,False,True,False,True,True,False,False,False
2,67.0,1,120.0,229.0,0,129.0,1,2.6,2.0,False,False,True,False,True,True,False,False,True
3,37.0,1,130.0,250.0,0,187.0,0,3.5,0.0,False,True,False,False,False,False,True,False,False
4,41.0,0,130.0,204.0,0,172.0,0,1.4,0.0,True,False,False,False,True,False,False,False,False


In [64]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)